# Episode 2 — Vectorized `jit`, Timing Matmul, and jaxpr

**Instructor notebook** · run top-to-bottom before recording.

Episode 1 built an eager affine forward. Here we **compile** with `jax.jit`, **batch** with `jax.vmap`, inspect **jaxprs**, and **time** eager vs compiled matmul.

| | |
|---|---|
| **Prereq** | Episode 1 — PRNG keys, nested `params`, broadcasting |
| **Next** | Episode 3 — `scan` vs Python loop (prefix sum) |

**JAX docs:** [Tracing](https://docs.jax.dev/en/latest/tracing.html) · [Key concepts — jaxpr & XLA](https://docs.jax.dev/en/latest/key-concepts.html) · [`jax.jit`](https://docs.jax.dev/en/latest/_autosummary/jax.jit.html) · [`jax.vmap`](https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html) · [`jax.make_jaxpr`](https://docs.jax.dev/en/latest/_autosummary/jax.make_jaxpr.html)


In [ ]:
import timeit

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import make_jaxpr


## What `jax.jit` is doing

**`jax.jit` does not "run Python faster."** On the first call with given array **shapes and dtypes**, JAX **traces** your function: array values are replaced by **tracers**, and each `jnp` op records a primitive into a **jaxpr** (JAX program). That jaxpr is **lowered** to **StableHLO**, then **XLA** optimizes and emits code for your device. Later calls with the **same shapes/dtypes** reuse the compiled executable; new shapes trigger **recompilation**.

Read: [Tracing](https://docs.jax.dev/en/latest/tracing.html) · [jaxpr](https://docs.jax.dev/en/latest/jaxpr.html)


## JIT limitations (preview)

Keep these in mind whenever you reach for `jit`:

1. **Only traced ops compile** — arbitrary Python (`for`, `print`, file I/O) does not magically speed up; only `jnp` / `lax` primitives recorded during tracing become XLA code.
2. **One executable per shape/dtype** — change `B` or matrix sizes and JAX **re-traces** (compile again).
3. **Pure functions** — no in-place array writes (`x[0] = 1` fails); use `.at[...].set(...)` and return the new array (Episode 0).
4. **Timing is tricky** — JAX dispatches asynchronously; call `jax.block_until_ready(...)` before stopping the clock on GPU/TPU.


In [ ]:
M, K, N = 512, 512, 512
key = jr.key(0)
key, k_a, k_b = jr.split(key, 3)
A = jr.normal(k_a, (M, K))
B = jr.normal(k_b, (K, N))


def matmul_fn(a, b):
    return jnp.dot(a, b)


C = matmul_fn(A, B)
jitted_matmul = jax.jit(matmul_fn)
C_jit = jitted_matmul(A, B)
print("match:", jnp.allclose(C, C_jit))


## Inspect jaxprs

Compare the primitives JAX records for eager trace vs a `jit`-wrapped function.


In [ ]:
print("eager trace:")
print(make_jaxpr(matmul_fn)(A, B))
print("jit trace:")
print(make_jaxpr(jitted_matmul)(A, B))


## Time eager vs `jit`

Warm up once, then time steady-state. The **first** call on fresh `jit(fn)` includes trace + compile.


In [ ]:
def time_ms(fn, *args, repeat=10):
    def run():
        jax.block_until_ready(fn(*args))

    return min(timeit.repeat(run, number=1, repeat=repeat)) * 1000


eager_ms = time_ms(matmul_fn, A, B)

fresh_jit = jax.jit(matmul_fn)
first_jit_ms = time_ms(fresh_jit, A, B)
for _ in range(3):
    fresh_jit(A, B)
steady_jit_ms = time_ms(fresh_jit, A, B)

print(f"eager:              {eager_ms:7.2f} ms")
print(f"jitted (1st call):  {first_jit_ms:7.2f} ms  # trace + compile")
print(f"jitted (steady):    {steady_jit_ms:7.2f} ms")


## Recompilation on new shapes

Call the same jitted function with different `(M, K, N)` — JAX builds a new executable.


In [ ]:
A_small = jr.normal(jr.key(1), (64, 64))
B_small = jr.normal(jr.key(2), (64, 64))

new_shape_ms = time_ms(jitted_matmul, A_small, B_small)
same_shape_ms = time_ms(jitted_matmul, A_small, B_small)
print(f"new shapes:  {new_shape_ms:.2f} ms")
print(f"same shapes: {same_shape_ms:.2f} ms")


## Batched matmul with `vmap`

Vectorize over a leading batch axis: `(B, M, K) @ (B, K, N) → (B, M, N)`.


In [ ]:
B_batch = 32
A_b = jr.normal(jr.key(3), (B_batch, 128, 128))
B_b = jr.normal(jr.key(4), (B_batch, 128, 128))

batched_matmul = jax.vmap(jnp.dot)
C_b = batched_matmul(A_b, B_b)

C_loop = jnp.stack([jnp.dot(A_b[i], B_b[i]) for i in range(B_batch)])
print("vmap matches loop:", jnp.allclose(C_b, C_loop))


## `jit` ∘ `vmap`

Compile the whole batched kernel in one shot.


In [ ]:
jitted_batched = jax.jit(jax.vmap(jnp.dot))
C_jit_b = jitted_batched(A_b, B_b)
print("jit(vmap) matches vmap:", jnp.allclose(C_b, C_jit_b))


## Batched affine forward (Episode 1)

`vmap` over the batch dimension of `x`; `params` is shared (`in_axes=(None, 0)`).


In [ ]:
def forward(params, x):
    W = params["affine"]["W"]
    b = params["affine"]["b"]
    return x @ W + b


d_in, d_out = 10, 10
params = {"affine": {"W": jr.normal(jr.key(5), (d_in, d_out)) * 0.1, "b": jnp.zeros(d_out)}}
x = jr.normal(jr.key(6), (B_batch, d_in))

batched_forward = jax.jit(jax.vmap(forward, in_axes=(None, 0)))
y_b = batched_forward(params, x)
print("y_b.shape:", y_b.shape)
print(make_jaxpr(batched_forward)(params, x))


---
## Exercise

1. `make_jaxpr` on `jax.jit(jax.vmap(forward, in_axes=(None, 0)))` for `B=64` — find a `dot_general` (or similar) line in the output.
2. `timeit` eager `vmap(forward)` vs `jit(vmap(forward))` after warmup.

*(Solution below.)*


In [ ]:
B_ex = 64
x_ex = jr.normal(jr.key(7), (B_ex, d_in))
eager_batched = jax.vmap(forward, in_axes=(None, 0))
jitted_batched_forward = jax.jit(jax.vmap(forward, in_axes=(None, 0)))

print(make_jaxpr(jitted_batched_forward)(params, x_ex))

eager_ms = time_ms(eager_batched, params, x_ex)
for _ in range(3):
    jitted_batched_forward(params, x_ex)
steady_ms = time_ms(jitted_batched_forward, params, x_ex)
print(f"eager vmap: {eager_ms:.2f} ms | jitted vmap (steady): {steady_ms:.2f} ms")


**Next:** Episode 3 — prefix sum with a Python loop vs `lax.scan` (no `jit`).
